Script generated by Alexa Ma

### import libraries

In [ ]:
# libraries
import numpy as np
import napari
import os
from skimage import io
import re
import nd2
import matplotlib.pyplot as plt
from skimage import data
from skimage.registration import phase_cross_correlation
from skimage.registration._phase_cross_correlation import _upsampled_dft
from scipy.ndimage import fourier_shift
import cv2
from scipy.ndimage import gaussian_filter, maximum_filter
from skimage.feature import peak_local_max
from skimage import data, filters, measure, morphology
from cellpose import models
import torch
from skimage.measure import label, regionprops
from skimage import measure
from skimage.morphology import label
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr
from scipy.fft import fft2, fftshift
from scipy.ndimage import zoom
from numpy.fft import fft2, fftshift

### functions
#### no need to make edits here - just run the cells!

In [ ]:
# In case the two images you want to correlate are offset in the x/y plane, this function determines the number of pixels one image is offset from the other as: (y pixels, x pixels)
# Code from: https://scikit-image.org/docs/0.23.x/auto_examples/registration/plot_register_translation.html

def shift_fix(image, offset_image):
    # pixel precision first
    shift, error, diffphase = phase_cross_correlation(image, offset_image)

    fig = plt.figure(figsize=(8, 3))
    ax1 = plt.subplot(1, 3, 1)
    ax2 = plt.subplot(1, 3, 2, sharex=ax1, sharey=ax1)
    ax3 = plt.subplot(1, 3, 3)

    ax1.imshow(image, cmap='gray')
    ax1.set_axis_off()
    ax1.set_title('Reference image')

    ax2.imshow(offset_image.real, cmap='gray')
    ax2.set_axis_off()
    ax2.set_title('Offset image')

    # Show the output of a cross-correlation to show what the algorithm is doing behind the scenes
    image_product = np.fft.fft2(image) * np.fft.fft2(offset_image).conj()
    cc_image = np.fft.fftshift(np.fft.ifft2(image_product))
    ax3.imshow(cc_image.real)
    ax3.set_axis_off()
    ax3.set_title("Cross-correlation")

    plt.show()

    print(f'Detected pixel offset (y, x): {shift}')

    # subpixel precision
    shift, error, diffphase = phase_cross_correlation(image, offset_image, upsample_factor=100)

    fig = plt.figure(figsize=(8, 3))
    ax1 = plt.subplot(1, 3, 1)
    ax2 = plt.subplot(1, 3, 2, sharex=ax1, sharey=ax1)
    ax3 = plt.subplot(1, 3, 3)

    ax1.imshow(image, cmap='gray')
    ax1.set_axis_off()
    ax1.set_title('Reference image')

    ax2.imshow(offset_image.real, cmap='gray')
    ax2.set_axis_off()
    ax2.set_title('Offset image')

    # Calculate the upsampled DFT, again to show what the algorithm is doing behind the scenes.  Constants correspond to calculated values in routine.
    # See source code for details.
    cc_image = _upsampled_dft(image_product, 150, 100, (shift * 100) + 75).conj()
    ax3.imshow(cc_image.real)
    ax3.set_axis_off()
    ax3.set_title("Supersampled XC sub-area")

    plt.show()

    print(f'Detected subpixel offset (y, x): {shift}')
    
    rounded_shift = [round(i) for i in shift]
    return rounded_shift

In [ ]:
# This function shifts one of the images to match the other by the amount specified by function shift_fix

def shift_image(image, shift_values):
    # Round the shift values to the nearest integer
    y_shift, x_shift = np.round(shift_values).astype(int)
    
    # Invert the y_shift because positive means up in the input, but down in numpy indexing
    y_shift = -y_shift
    
    # Create a new array of zeros with the same shape and dtype as the input image
    shifted_image = np.zeros_like(image)
    
    # Determine the slice of the original image to be copied
    if y_shift >= 0:
        src_y_start, src_y_end = 0, image.shape[0] - y_shift
        dst_y_start, dst_y_end = y_shift, image.shape[0]
    else:
        src_y_start, src_y_end = -y_shift, image.shape[0]
        dst_y_start, dst_y_end = 0, image.shape[0] + y_shift
    
    if x_shift >= 0:
        src_x_start, src_x_end = x_shift, image.shape[1]
        dst_x_start, dst_x_end = 0, image.shape[1] - x_shift
    else:
        src_x_start, src_x_end = 0, image.shape[1] + x_shift
        dst_x_start, dst_x_end = -x_shift, image.shape[1]
    
    # Copy the appropriate slice of the original image to the new image
    shifted_image[dst_y_start:dst_y_end, dst_x_start:dst_x_end] = \
        image[src_y_start:src_y_end, src_x_start:src_x_end]
    
    return shifted_image

In [ ]:
# Cellpose implementation
# This function is exclusively for cells/nuclei. It uses the cellpose algorithm to segment out the nucleus for analysis.

def implement_cellPose(model_type, estimated_diameter, flow_threshold, cellprob_threshold, image_slice):

    device = torch.device("cuda:0" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 8000000000 else "cpu")
    device_type = device.type
    gpu = True if "cuda" in str(device_type) else False

    # https://github.com/MouseLand/cellpose/blob/main/cellpose/models.py
    model = models.Cellpose(
                model_type = model_type,
                device = device,
                gpu = gpu)

    estimated_diameter = estimated_diameter #estimated size of cells (pixels)

    # picked some image to test on 
    #sum_image_smoth = image[18][1]
    #sum_image_smoth2 = io.imread('D:/Augusto/Images/ConfocalImaging/20221115/mCherry-Nup98nt1833-KDM5Ant4666-spy650-640nm30_1/mCherry-Nup98nt1833-KDM5Ant4666-spy650-640nm30_1_MMStack_5-Pos000_000.ome.tif')[1][25] 

    sum_image_smoth = image_slice
    batch = [sum_image_smoth]

    #flow_threshold 0.75
    masks, flows, styles, diams = model.eval(batch,
                                                 diameter = estimated_diameter,
                                                 flow_threshold = flow_threshold,
                                                 cellprob_threshold = cellprob_threshold,
                                                 channels = None,
                                                 tile = False)
    
    return masks, flows, styles, diams, batch

In [ ]:
# If there are multiple nuclei in an image, this function separates out the individual masks/nuclei and returns a list of individual masks.

def separate_blobs_and_apply_masks(mask):
    # Label the blobs in the mask
    labeled_mask = label(mask)
    
    # Get properties of the labeled regions
    regions = measure.regionprops(labeled_mask)
    
    # Create separate masks for each blob
    separate_masks = []
    for region in regions:
        blob_mask = np.zeros_like(mask, dtype=bool)
        blob_mask[tuple(region.coords.T)] = True
        separate_masks.append(blob_mask)
    
    return separate_masks

In [ ]:
# This function applies the individual masks to the two images you want to correlate, generating a dictionary of masked individual nuclei data for each image (h3k4me3 and puncta).

def apply_masks(separate_masks, shifted_h3k4me3, puncta):
    h3k4me3_finals = {}
    puncta_finals = {}
    
    for counter, mask in enumerate(separate_masks, start=1):
        h3k4me3_finals[f'h3k4me3_final_{counter}'] = np.where(mask, shifted_h3k4me3, 0)
        puncta_finals[f'puncta_final_{counter}'] = np.where(mask, puncta, 0)
    
    return h3k4me3_finals, puncta_finals

In [ ]:
# This function is for image plotting aesthetics.

def crop_with_margin(image, margin=100):
    # Find non-zero pixels (i.e. the cluster)
    nonzero_coords = np.argwhere(image > 0)
    
    if nonzero_coords.size == 0:
        # No white cluster found — return original image
        return image.copy()
    
    # Bounding box of the white cluster
    min_row, min_col = nonzero_coords.min(axis=0)
    max_row, max_col = nonzero_coords.max(axis=0)

    # Add margin, clamp within image bounds
    start_row = max(min_row - margin, 0)
    end_row = min(max_row + margin + 1, image.shape[0])
    start_col = max(min_col - margin, 0)
    end_col = min(max_col + margin + 1, image.shape[1])

    # Crop image
    cropped = image[start_row:end_row, start_col:end_col]

    # Make square by padding
    height, width = cropped.shape
    size = max(height, width)
    pad_height = size - height
    pad_width = size - width

    # Pad with black (0) pixels to make square
    pad_top = pad_height // 2
    pad_bottom = pad_height - pad_top
    pad_left = pad_width // 2
    pad_right = pad_width - pad_left

    square_cropped = np.pad(cropped, 
                            pad_width=((pad_top, pad_bottom), (pad_left, pad_right)), 
                            mode='constant', constant_values=0)
    
    return square_cropped

### classes
#### criteria:
- make sure that Nup98-KDM5A (puncta) is in the 2nd channel and H3K4me3 is in the 3rd channel. otherwise adjust accordingly in the Image class methods: set_raw_puncta, and set_raw_h3k4me3.
- make sure that file inputs are in .nd2 format and all within a single folder.
- for labeling/identification purposes, make sure that there is an appropriate 2 digit number in the name of each data file like: "U2OS-Nup98-KDM5A-mEGFP-H3K4me3-AF546_05.nd2"

#### afterward - just run the cells!

In [ ]:
# This class is for accessing files from the appropriate folder. No edits needed for this code - just run!

class Image_Folder:
    def __init__(self, directory):
        self.directory = directory
        self.file_numbers = []  # To store file numbers

    def load(self):
        nd2_files = [f for f in os.listdir(self.directory) if f.endswith('.nd2')]

        all_data = {}
        for file_name in nd2_files:
            file_path = os.path.join(self.directory, file_name)
            match = re.search(r'(\d{2})\.nd2$', file_name)
            if match:
                file_number = match.group(1)
            else:
                file_number = "00"

            self.file_numbers.append(file_number)  # Store the file number in the instance

            data = nd2.imread(file_path)
            #var_name = f"array_{file_number}"

            globals()[file_number] = data
            all_data[file_number] = data

        return all_data

In [ ]:
# This class is for choosing a Z slice, making any necessary pixel shifts, and masking cells in an image. No edits needed for this code - just run!

class Image:
    z_center = 0
    puncta = 0
    h3k4me3 = 0
    masks = 0

    #cellpose parameters
    model_type = 'cyto' #nuclei', #'cyto', # cyto2 #models provided by cellpose
    flow_threshold = 0.7 #flow_threshold 0.7. Higher threshold enables more permissive cell detection
    cellprob_threshold = 0.35 #0.35 works well 
    contrast_stretching = 0 #takes percentiles on both edges of histogram (eg., for 5, 5th and 95th percentiles) values 0-100
    estimated_diameter = 500
    
    def __init__(self, data):  # constructor
        self.data = data # complete image data
    
    def set_center(self):
        for z in range(data.shape[0]):
            img = data[z, 2, :, :]  # Looking at the H3K4me3 channel (2)
        
            plt.figure(figsize=(6, 6))
            plt.imshow(img, cmap='gray')
            plt.title(f"Z-slice {z}")
            plt.axis("off")
            plt.show()
            
        while True:
            try:
                Image.z_center = int(input("Enter the z-slice # you want to analyze: "))
                break
            except ValueError:
                print("Invalid input. Please enter an integer.")

    def set_raw_puncta(self):
        Image.puncta = self.data[Image.z_center, 1, :, :]
        
    def set_raw_h3k4me3(self):
        Image.h3k4me3 = self.data[Image.z_center, 2, :, :]

    def shift_h3k4me3(self):
        sharp_puncta = Image.puncta * 200
        round_shift = shift_fix(Image.h3k4me3, sharp_puncta)
        Image.h3k4me3 = shift_image(Image.h3k4me3, round_shift)

    def set_diameter(self):
        smoothed_h3k4me3 = gaussian_filter(Image.h3k4me3, sigma=0.75)
        smoothed_puncta = gaussian_filter(Image.puncta, sigma=0.75)
        
        while True:
            try:
                while True:
                    try:
                        Image.estimated_diameter = int(input("Enter the estimated diameter you want for Cellpose: "))
                        break
                    except ValueError:
                        print("Invalid input. Please enter an integer.")
                
                masks, flows, styles, diams, batch = implement_cellPose(
                    Image.model_type,
                    Image.estimated_diameter,
                    Image.flow_threshold,
                    Image.cellprob_threshold,
                    smoothed_h3k4me3
                )
                
                # Visualize
                plt.figure(figsize=(8, 8))
                plt.imshow(Image.h3k4me3, cmap='gray')  # Adjust the color map as needed
                plt.imshow(masks[0], cmap='jet', alpha=0.4)  # Adjust the colormap and alpha as needed
                plt.title(f'Overlay of Cellpose Mask on H3K4me3 (Diameter {Image.estimated_diameter})')
                plt.axis('off')  # Hide the axes for better visualization
                plt.show()
                
                satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                if satisfied == 'y':
                    break
            except Exception as e:
                print(f"Something went wrong: {e}. Please try again.")

        Image.masks = masks

In [ ]:
# This class is for isolating individual nuclei in the image and creating all relevant analysis plots. No edits needed for this code - just run!

class Cell:
    def __init__(self, h3k4me3_data, puncta_data, masks, key, cell_type):  # constructor
        self.h3k4me3_data = h3k4me3_data
        self.puncta_data = puncta_data
        self.masks = masks
        self.key = key # string of cell's image's file number
        self.cell_type = cell_type # string of type of cell (mEGFP, mCherry, etc)
        self.pearson_rs = []
        self.ious = []
        self.expressions = []
        self.h3k4me3_by_cell = 0
        self.puncta_by_cell = 0
        self.crossing_freqs = []
        self.frc_at_target = [] 
        self.separate_masks = []

    def separate_cells(self):
        mask = self.masks[0]
        self.separate_masks = separate_blobs_and_apply_masks(mask)
        self.h3k4me3_by_cell, self.puncta_by_cell = apply_masks(self.separate_masks, self.h3k4me3_data, self.puncta_data)

    def use_existing_masks(self, cellpose_storage_masks):
        self.separate_masks = cellpose_storage_masks[self.key]
        self.h3k4me3_by_cell, self.puncta_by_cell = apply_masks(cellpose_storage_masks[self.key], self.h3k4me3_data, self.puncta_data)

    # PEARSON VS. EXPRESSION PLOTS
    def just_exp_pearson(self, img_dict_puncta):
        self.pearson_rs = []
        self.ious = []
        self.expressions = []
        final_puncta_lst = list(img_dict_puncta.values()) 
        final_h3k4me3_lst = list(self.h3k4me3_by_cell.values())

        for index in range(len(final_puncta_lst)):
            print("Index: ", index)

            # Normalize images
            puncta_norm = (final_puncta_lst[index] - final_puncta_lst[index].min()) / (final_puncta_lst[index].max() - final_puncta_lst[index].min())
            h3k4me3_norm = (final_h3k4me3_lst[index] - final_h3k4me3_lst[index].min()) / (final_h3k4me3_lst[index].max() - final_h3k4me3_lst[index].min())

            # Unnormalized images
            #puncta_norm = final_puncta_lst[index]
            #h3k4me3_norm = final_h3k4me3_lst[index]
            
            # Flatten images
            x_coords = puncta_norm.flatten()
            y_coords = h3k4me3_norm.flatten()

            # Filtering out the points where y (h3k4me3) are 0
            mask = (y_coords != 0)
            x_filtered = x_coords[mask]
            y_filtered = y_coords[mask]
            
            # Filtering out the points where x (puncta) are 0
            mask_2 = (x_filtered != 0)
            x_filtered = x_filtered[mask_2]
            y_filtered = y_filtered[mask_2]
            
            # Get Pearson's coefficient and fit a line using numpy.polyfit (degree 1 for linear fit)
            x = x_filtered
            y = y_filtered

            r, p_value = pearsonr(x, y)  # (x, y) = (Nup98-KDM5A-meGFP, H3K4me3)
            print("Min/Max of normalized puncta image: (", x.min(), x.max(), ")")
            print("Min/Max of normalized H3K4me3 image: (", y.min(), y.max(), ")")
            print(f'Pearson correlation coefficient (r): {r:.4f}')
            slope, intercept = np.polyfit(x, y, 1)

            # Expression
            puncta_array = final_puncta_lst[index]
            puncta_mask = self.separate_masks[index]
            masked_puncta = puncta_array[puncta_mask.astype(bool)]
            expression = np.mean(masked_puncta)
            print(f"Expression: {expression}")
            
            self.pearson_rs.append(r)
            self.expressions.append(expression)

    # LINEAR 2D HISTOGRAMS WITHOUT MANUAL SCALING 
    def make_linear_unscaled_plots(self):
        
        # generate contrasted images
        while True:
            try:
                contrast_puncta = {}
                contrast_h3k4me3 = {}
                
                puncta_contrast_coeff = float(input("Enter the contrast coefficient you want for puncta: "))
                h3k4me3_contrast_coeff = float(input("Enter the contrast coefficient you want for h3k4me3: "))

                for counter, puncta_final in enumerate(self.puncta_by_cell.values(), start=1):
                    contrast = np.clip(puncta_final * puncta_contrast_coeff, 0, 255).astype(np.uint8)
                    contrast_image = crop_with_margin(contrast, margin=150)
                    contrast_puncta[f'contrast_puncta_{counter}'] = contrast_image/contrast_image.max()
                
                for counter, h3k4me3_final in enumerate(self.h3k4me3_by_cell.values(), start=1):
                    contrast = np.clip(h3k4me3_final * h3k4me3_contrast_coeff, 0, 255).astype(np.uint8)
                    contrast_image = crop_with_margin(contrast, margin=150)
                    contrast_h3k4me3[f'contrast_h3k4me3_{counter}'] = contrast_image/contrast_image.max()
                
                # Images
                contrast_puncta_images = list(contrast_puncta.values())
                contrast_h3k4me3_images = list(contrast_h3k4me3.values())
                
                # Set up the figure size
                plt.figure(figsize=(5, 5))
                
                # Loop through each image and plot
                for i, img in enumerate(contrast_puncta_images):
                    plt.figure()  # Create a new figure for each image
                    plt.imshow(img, cmap='gray')  # Show the image
                    plt.title(f'Contrast Puncta {i+1}')  # Optional: Set title for each image
                    plt.axis('off')  # Hide the axis
                    plt.show()  # Display the image

                # Set up the figure size
                plt.figure(figsize=(5, 5))
                
                # Loop through each image and plot
                for i, img in enumerate(contrast_h3k4me3_images):
                    plt.figure()  # Create a new figure for each image
                    plt.imshow(img, cmap='gray')  # Show the image
                    plt.title(f'Contrast H3K4me3 {i+1}')  # Optional: Set title for each image
                    plt.axis('off')  # Hide the axis
                    plt.show()  # Display the image
                
                # Show all images
                plt.tight_layout()  # Optional: Adjust spacing between plots
                plt.show()
                
                satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                if satisfied == 'y':
                    break
            except Exception as e:
                print(f"Something went wrong: {e}. Please try again.")

        # heatmap plots
        final_puncta_lst = list(self.puncta_by_cell.values()) 
        final_h3k4me3_lst = list(self.h3k4me3_by_cell.values())

        for index in range(len(final_puncta_lst)):
            print(index)
            while True:
                try:     
                    x_coords = final_puncta_lst[index].flatten()
                    y_coords = final_h3k4me3_lst[index].flatten()
                    
                    # Filtering out the points where y (h3k4me3) are 0
                    mask = (y_coords != 0)
                    x_filtered = x_coords[mask]
                    y_filtered = y_coords[mask]
                    
                    # Filtering out the points where x (puncta) are 0
                    mask_2 = (x_filtered != 0)
                    x_filtered = x_filtered[mask_2]
                    y_filtered = y_filtered[mask_2]
                    
                    # Get Pearson's coefficient and fit a line using numpy.polyfit (degree 1 for linear fit)
                    x = x_filtered
                    y = y_filtered
        
                    r, p_value = pearsonr(x, y)  # (x, y) = (Nup98-KDM5A, H3K4me3)
        
                    print(f'Pearson correlation coefficient (r): {r:.4f}')
                    print(f'p-value: {p_value:.4e}')
                    slope, intercept = np.polyfit(x, y, 1)
        
                    # Expression
                    puncta_array = final_puncta_lst[index]
                    puncta_mask = self.separate_masks[index]
                    masked_puncta = puncta_array[puncta_mask.astype(bool)]
                    expression = np.mean(masked_puncta)
                    print(f"Expression: {expression}")
                    
                    # Generate line values
                    line_y = slope * x + intercept
        
                    # Plotting the coordinates using a 2D histogram
                    fig, axs = plt.subplots(1, 3, figsize=(22, 6))
                    
                    # Plot the first array
                    a = axs[0].imshow(contrast_puncta_images[index], cmap='gray')
                    cbar_0 = fig.colorbar(a, ax=axs[0], orientation='vertical')
                    cbar_0.ax.tick_params(labelsize=14)
                    axs[0].tick_params(axis='both', which='major', labelsize=14)
                    axs[0].set_title(self.cell_type, fontsize=15)
                    axs[0].set_xlabel('Pixels', fontsize=14)
                    axs[0].set_ylabel('Pixels', fontsize=14)
                    
                    # Plot the second array
                    b = axs[1].imshow(contrast_h3k4me3_images[index], cmap='gray')
                    cbar_1 = fig.colorbar(b, ax=axs[1], orientation='vertical')
                    cbar_1.ax.tick_params(labelsize=14)
                    axs[1].tick_params(axis='both', which='major', labelsize=14)
                    axs[1].set_title('H3K4me3', fontsize=15)
                    axs[1].set_xlabel('Pixels', fontsize=14)
                    axs[1].set_ylabel('Pixels', fontsize=14)

                    # Plot the third array (LINEAR)
                    bin_number = int(input("Enter the bin number for histograms: "))

                    h = axs[2].hist2d(x_filtered, y_filtered, bins=[bin_number, bin_number], cmap='plasma', norm=LogNorm())
                    cbar_2 = fig.colorbar(h[3], ax=axs[2])
                    cbar_2.ax.tick_params(labelsize=14)
                    
                    axs[2].tick_params(axis='both', which='major', labelsize=14)
                    axs[2].set_ylabel('H3K4me3 Intensities', fontsize=14)
                    axs[2].set_xlabel(f'{self.cell_type} Intensities', fontsize=14)
                    axs[2].set_title(f'Cell {key}-{index:01d}: {self.cell_type} vs. H3K4me3', fontsize=15)
                    
                    # Optional: Overlay a line, ensure it's positive and log-scaled too
                    #axs[2].plot(x, line_y, color='black', linestyle='--')

                    # Display the plots
                    plt.tight_layout()  # Adjust the layout to avoid overlap
                    fig.savefig(f'2Dhist_{self.cell_type}_{self.key}_{index:01d}.png', bbox_inches='tight', dpi=300)
                    plt.show()
                            
                    satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                    if satisfied == 'y':
                        self.pearson_rs.append(r)
                        self.expressions.append(expression)
                        break
                except Exception as e:
                    print(f"Something went wrong: {e}. Please try again.")

    # LINEAR 2D HISTOGRAMS WITH MANUAL SCALING
    def make_linear_plots(self):

        # generate contrasted images
        while True:
            try:
                contrast_puncta = {}
                contrast_h3k4me3 = {}
                
                puncta_contrast_coeff = float(input("Enter the contrast coefficient you want for puncta: "))
                h3k4me3_contrast_coeff = float(input("Enter the contrast coefficient you want for h3k4me3: "))

                for counter, puncta_final in enumerate(self.puncta_by_cell.values(), start=1):
                    contrast = np.clip(puncta_final * puncta_contrast_coeff, 0, 255).astype(np.uint8)
                    contrast_image = crop_with_margin(contrast, margin=150)
                    contrast_puncta[f'contrast_puncta_{counter}'] = contrast_image/contrast_image.max()
                
                for counter, h3k4me3_final in enumerate(self.h3k4me3_by_cell.values(), start=1):
                    contrast = np.clip(h3k4me3_final * h3k4me3_contrast_coeff, 0, 255).astype(np.uint8)
                    contrast_image = crop_with_margin(contrast, margin=150)
                    contrast_h3k4me3[f'contrast_h3k4me3_{counter}'] = contrast_image/contrast_image.max()
                
                # Images
                contrast_puncta_images = list(contrast_puncta.values())
                contrast_h3k4me3_images = list(contrast_h3k4me3.values())
                
                # Set up the figure size
                plt.figure(figsize=(5, 5))
                
                # Loop through each image and plot
                for i, img in enumerate(contrast_puncta_images):
                    plt.figure()  # Create a new figure for each image
                    plt.imshow(img, cmap='gray')  # Show the image
                    plt.title(f'Contrast Puncta {i+1}')  # Optional: Set title for each image
                    plt.axis('off')  # Hide the axis
                    plt.show()  # Display the image
                
                # Set up the figure size
                plt.figure(figsize=(5, 5))
                
                # Loop through each image and plot
                for i, img in enumerate(contrast_h3k4me3_images):
                    plt.figure()  # Create a new figure for each image
                    plt.imshow(img, cmap='gray')  # Show the image
                    plt.title(f'Contrast H3K4me3 {i+1}')  # Optional: Set title for each image
                    plt.axis('off')  # Hide the axis
                    plt.show()  # Display the image
                
                # Show all images
                plt.tight_layout()  # Optional: Adjust spacing between plots
                plt.show()
                
                satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                if satisfied == 'y':
                    break
            except Exception as e:
                print(f"Something went wrong: {e}. Please try again.")

        # heatmap plots
        final_puncta_lst = list(self.puncta_by_cell.values()) 
        final_h3k4me3_lst = list(self.h3k4me3_by_cell.values())

        for index in range(len(final_puncta_lst)):
            print(index)
            while True:
                try:
                    x_coords = final_puncta_lst[index].flatten()
                    y_coords = final_h3k4me3_lst[index].flatten()
                    
                    # Filtering out the points where y (h3k4me3) are 0
                    mask = (y_coords != 0)
                    x_filtered = x_coords[mask]
                    y_filtered = y_coords[mask]
                    
                    # Filtering out the points where x (puncta) are 0
                    mask_2 = (x_filtered != 0)
                    x_filtered = x_filtered[mask_2]
                    y_filtered = y_filtered[mask_2]
                    
                    # Get Pearson's coefficient and fit a line using numpy.polyfit (degree 1 for linear fit)
                    x = x_filtered
                    y = y_filtered
        
                    r, p_value = pearsonr(x, y)  # (x, y) = (Nup98-KDM5A, H3K4me3)
                    #self.pearson_p = p
        
                    print(f'Pearson correlation coefficient (r): {r:.4f}')
                    print(f'p-value: {p_value:.4e}')
                    slope, intercept = np.polyfit(x, y, 1)
        
                    # Expression
                    puncta_array = final_puncta_lst[index]
                    puncta_mask = self.separate_masks[index]
                    masked_puncta = puncta_array[puncta_mask.astype(bool)]
                    expression = np.mean(masked_puncta)
                    print(f"Expression: {expression}")
                    
                    # Generate line values
                    line_y = slope * x + intercept
        
                    # Plotting the coordinates using a 2D histogram
                    fig, axs = plt.subplots(1, 3, figsize=(22, 6))
                    
                    # Plot the first array
                    a = axs[0].imshow(contrast_puncta_images[index], cmap='gray')
                    cbar_0 = fig.colorbar(a, ax=axs[0], orientation='vertical')
                    cbar_0.ax.tick_params(labelsize=14)
                    axs[0].tick_params(axis='both', which='major', labelsize=14)
                    axs[0].set_title(self.cell_type, fontsize=15)
                    axs[0].set_xlabel('Pixels', fontsize=14)
                    axs[0].set_ylabel('Pixels', fontsize=14)
                    
                    # Plot the second array
                    b = axs[1].imshow(contrast_h3k4me3_images[index], cmap='gray')
                    cbar_1 = fig.colorbar(b, ax=axs[1], orientation='vertical')
                    cbar_1.ax.tick_params(labelsize=14)
                    axs[1].tick_params(axis='both', which='major', labelsize=14)
                    axs[1].set_title('H3K4me3', fontsize=15)
                    axs[1].set_xlabel('Pixels', fontsize=14)
                    axs[1].set_ylabel('Pixels', fontsize=14)

                    # Plot the third array (LINEAR, SAME AXIS SCALING)
                    bin_number = int(input("Enter the bin number for histograms: "))
                    
                    x_min, x_max = 0, 500
                    y_min, y_max = 0, 1500

                    h = axs[2].hist2d(x_filtered, y_filtered, bins=[bin_number, bin_number], cmap='plasma', norm=LogNorm())
                    cbar_2 = fig.colorbar(h[3], ax=axs[2])
                    cbar_2.ax.tick_params(labelsize=14)
                    
                    axs[2].tick_params(axis='both', which='major', labelsize=14)
                    axs[2].set_ylabel('H3K4me3 Intensities', fontsize=14)
                    axs[2].set_xlabel(f'{self.cell_type} Intensities', fontsize=14)
                    axs[2].set_title(f'Cell {key}-{index:01d}: {self.cell_type} vs. H3K4me3', fontsize=15)
                    axs[2].set_xlim(x_min, x_max)
                    axs[2].set_ylim(y_min, y_max)
                    
                    # Optional: Overlay a line, ensure it's positive and log-scaled too
                    #axs[2].plot(x, line_y, color='black', linestyle='--')

                    # Display the plots
                    plt.tight_layout()  # Adjust the layout to avoid overlap
                    fig.savefig(f'2Dhist_lin_{self.cell_type}_{self.key}_{index:01d}.png', bbox_inches='tight', dpi=300)
                    plt.show()
                            
                    satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                    if satisfied == 'y':
                        #self.pearson_rs.append(r)
                        #self.expressions.append(expression)
                        break
                except Exception as e:
                    print(f"Something went wrong: {e}. Please try again.")
                    
    # LOG-SCALED 2D HISTOGRAMS
    def make_log_plots(self):

        # generate contrasted images
        while True:
            try:
                contrast_puncta = {}
                contrast_h3k4me3 = {}
                
                puncta_contrast_coeff = float(input("Enter the contrast coefficient you want for puncta: "))
                h3k4me3_contrast_coeff = float(input("Enter the contrast coefficient you want for h3k4me3: "))

                for counter, puncta_final in enumerate(self.puncta_by_cell.values(), start=1):
                    contrast = np.clip(puncta_final * puncta_contrast_coeff, 0, 255).astype(np.uint8)
                    contrast_image = crop_with_margin(contrast, margin=150)
                    contrast_puncta[f'contrast_puncta_{counter}'] = contrast_image/contrast_image.max()
                
                for counter, h3k4me3_final in enumerate(self.h3k4me3_by_cell.values(), start=1):
                    contrast = np.clip(h3k4me3_final * h3k4me3_contrast_coeff, 0, 255).astype(np.uint8)
                    contrast_image = crop_with_margin(contrast, margin=150)
                    contrast_h3k4me3[f'contrast_h3k4me3_{counter}'] = contrast_image/contrast_image.max()
                
                # Images
                contrast_puncta_images = list(contrast_puncta.values())
                contrast_h3k4me3_images = list(contrast_h3k4me3.values())
                
                # Set up the figure size
                plt.figure(figsize=(5, 5))
                
                # Loop through each image and plot
                for i, img in enumerate(contrast_puncta_images):
                    plt.figure()  # Create a new figure for each image
                    plt.imshow(img, cmap='gray')  # Show the image
                    plt.title(f'Contrast Puncta {i+1}')  # Optional: Set title for each image
                    plt.axis('off')  # Hide the axis
                    plt.show()  # Display the image
                
                # Set up the figure size
                plt.figure(figsize=(5, 5))
                
                # Loop through each image and plot
                for i, img in enumerate(contrast_h3k4me3_images):
                    plt.figure()  # Create a new figure for each image
                    plt.imshow(img, cmap='gray')  # Show the image
                    plt.title(f'Contrast H3K4me3 {i+1}')  # Optional: Set title for each image
                    plt.axis('off')  # Hide the axis
                    plt.show()  # Display the image
                
                # Show all images
                plt.tight_layout()  # Optional: Adjust spacing between plots
                plt.show()
                
                satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                if satisfied == 'y':
                    break
            except Exception as e:
                print(f"Something went wrong: {e}. Please try again.")

        # heatmap plots
        final_puncta_lst = list(self.puncta_by_cell.values()) 
        final_h3k4me3_lst = list(self.h3k4me3_by_cell.values())

        for index in range(len(final_puncta_lst)):
            print(index)
            while True:
                try:
                    x_coords = final_puncta_lst[index].flatten()
                    y_coords = final_h3k4me3_lst[index].flatten()
                    
                    # Filtering out the points where y (h3k4me3) are 0
                    mask = (y_coords != 0)
                    x_filtered = x_coords[mask]
                    y_filtered = y_coords[mask]
                    
                    # Filtering out the points where x (puncta) are 0
                    mask_2 = (x_filtered != 0)
                    x_filtered = x_filtered[mask_2]
                    y_filtered = y_filtered[mask_2]
                    
                    # Get Pearson's coefficient and fit a line using numpy.polyfit (degree 1 for linear fit)
                    x = x_filtered
                    y = y_filtered
        
                    r, p_value = pearsonr(x, y)  # (x, y) = (Nup98-KDM5A, H3K4me3)
                    #self.pearson_p = p_value
        
                    print(f'Pearson correlation coefficient (r): {r:.4f}')
                    print(f'p-value: {p_value:.4e}')
                    slope, intercept = np.polyfit(x, y, 1)
        
                    # Expression
                    puncta_array = final_puncta_lst[index]
                    puncta_mask = self.separate_masks[index]
                    masked_puncta = puncta_array[puncta_mask.astype(bool)]
                    expression = np.mean(masked_puncta)
                    print(f"Expression: {expression}")
                    
                    # Generate line values
                    line_y = slope * x + intercept
        
                    # Plotting the coordinates using a 2D histogram
                    fig, axs = plt.subplots(1, 3, figsize=(22, 6))
                    
                    # Plot the first array
                    a = axs[0].imshow(contrast_puncta_images[index], cmap='gray')
                    cbar_0 = fig.colorbar(a, ax=axs[0], orientation='vertical')
                    cbar_0.ax.tick_params(labelsize=14)
                    axs[0].tick_params(axis='both', which='major', labelsize=14)
                    axs[0].set_title(self.cell_type, fontsize=15)
                    axs[0].set_xlabel('Pixels', fontsize=14)
                    axs[0].set_ylabel('Pixels', fontsize=14)
                    
                    # Plot the second array
                    b = axs[1].imshow(contrast_h3k4me3_images[index], cmap='gray')
                    cbar_1 = fig.colorbar(b, ax=axs[1], orientation='vertical')
                    cbar_1.ax.tick_params(labelsize=14)
                    axs[1].tick_params(axis='both', which='major', labelsize=14)
                    axs[1].set_title('H3K4me3', fontsize=15)
                    axs[1].set_xlabel('Pixels', fontsize=14)
                    axs[1].set_ylabel('Pixels', fontsize=14)
                    
                    # Plot the third array (LOG NORMALIZED, SAME AXIS SCALING)
                    bin_number = int(input("Enter the bin number for histograms: "))
                    x_min, x_max = 1, 500
                    y_min, y_max = 1, 1500
                    binning = np.logspace(np.log10(x_min), np.log10(x_max), bin_number)

                    # Histogram with log-normalized bin counts
                    h = axs[2].hist2d(
                        x, y,
                        bins=[binning, binning],
                        cmap='plasma',
                        norm=LogNorm()
                    )

                    # Log scale for axes
                    axs[2].set_xscale('log')
                    axs[2].set_yscale('log')
                    
                    # Axis limits and ticks
                    axs[2].set_xlim(x_min, x_max)
                    axs[2].set_ylim(y_min, y_max)

                    axs[2].tick_params(axis='both', which='major', labelsize=14)
                    axs[2].set_ylabel('H3K4me3 Intensities', fontsize=14)
                    axs[2].set_xlabel(f'{self.cell_type} Intensities', fontsize=14)
                    axs[2].set_title(f'Cell {key}-{index:01d}: {self.cell_type} vs. H3K4me3', fontsize=15)
                    
                    # Colorbar
                    cbar_2 = fig.colorbar(h[3], ax=axs[2])
                    cbar_2.ax.tick_params(labelsize=14)
                    
                    # Optional: Overlay a line, ensure it's positive and log-scaled too
                    #axs[2].plot(x, line_y, color='black', linestyle='--')

                    # Display the plots
                    plt.tight_layout()  # Adjust the layout to avoid overlap
                    fig.savefig(f'2Dhist_log_{self.cell_type}_{self.key}_{index:01d}.png', bbox_inches='tight', dpi=300)
                    plt.show()
                            
                    satisfied = input("Are you satisfied with the result? (y/n): ").strip().lower()
                    if satisfied == 'y':
                        #self.pearson_rs.append(r)
                        #self.expressions.append(expression)
                        break
                except Exception as e:
                    print(f"Something went wrong: {e}. Please try again.")

### implementation
#### steps:
1.  in the "Loading files" cells, insert the path to the folder containing your appropriated formatted data files.
2.  run the "EMPTY DICTIONARIES" cell to define where we will store our values for any deeper analysis.
3.  run the "FILLING UP DICTIONARIES" cell to fill in dictionaries storing Pearson R values, expression levels, and cellpose masks.
4.  run the "LINEAR 2D HISTOGRAM PLOTS" cell to generate 2D histograms for each cell/nucleus with either manually scaled and unscaled axes.

In [ ]:
# Loading files (if the data is new)
folder = Image_Folder("/Users/alexama/Desktop/research/huang lab/FINALIZED WORK/DATA/20250111_mEGFP/Nup98-KDM5A-mEGFP-H3K4me3-AF647")
full_data = folder.load()
print(folder.file_numbers)  # This gives you a list of all file_numbers
print(full_data.keys())

In [ ]:
# EMPTY DICTIONARIES

# pearsons & exp
pearson_dict = {} # keys are file numbers, values are lists with corresponding Pearson r's
expression_dict = {}

In [ ]:
# FILLING UP DICTIONARIES
# This cell fills up the pearson_dict, expression_dict, and cellpose_mask_storage for keeping Pearson R values, expression levels and cellpose masks for each cell

cell_type = input("Enter the cell type (e.g., 'Nup98-KDM5A-mEGFP', 'Nup98-KDM5Amut-mEGFP', etc.): ").strip()
cellpose_mask_storage = {}
for key in folder.file_numbers:
    print(key)
    data = full_data[key]
    image = Image(data)
    #viewer.add_image(data)

    # processing image
    image.set_center()
    image.set_raw_puncta()
    image.set_raw_h3k4me3()
    #image.shift_h3k4me3() # uncomment this line if you want to correct for x-y shifts between channels
    image.set_diameter()

    # processing cells
    # for mEGFP
    cell = Cell(image.h3k4me3, image.puncta, image.masks, key, cell_type)
    # for mCherry
    #cell = Cell(image.h3k4me3 / 0.23, image.puncta / 0.23, image.masks, key, cell_type) # uncomment this line if cells are Nup98-KDM5A-mCherry
    cell.separate_cells()
    cellpose_mask_storage[key] = cell.separate_masks

    puncta = cell.puncta_by_cell
    cell.just_exp_pearson(puncta)

    r_value = cell.pearson_rs
    pearson_dict[key] = r_value

    exp_value = cell.expressions
    expression_dict[key] = exp_value

In [ ]:
# Export the cellpose_mask_storage dictionary as a txt file to keep and reuse all your masks

# Ensure that cellpose_mask_storage stored all the masks
print(cellpose_mask_storage.keys())
print(cellpose_mask_storage)

# Export cellpose_mask_storage as a txt file to use with JSON
import json

# Convert arrays to lists so JSON can handle them
json_ready = {k: [arr.tolist() for arr in v] for k, v in cellpose_mask_storage.items()}

# Save to TXT
with open("cellpose_mask_storage.txt", "w") as f:
    json.dump(json_ready, f)

In [ ]:
# Import cellpose masks from txt file to a dictionary
import json
with open('cellpose_mask_storage.txt', 'r') as f:
    loaded = json.load(f)

# Convert lists back to numpy arrays
masks_loaded = {k: [np.array(arr) for arr in v] for k, v in loaded.items()}
print(masks_loaded)

In [ ]:
#### LINEAR 2D HISTOGRAM PLOTS

for key in folder.file_numbers:
    print(key)
    data = full_data[key]
    image = Image(data)
    #viewer.add_image(data)

    # processing image
    image.set_center()
    image.set_raw_puncta()
    image.set_raw_h3k4me3()
    #image.shift_h3k4me3() # uncomment this line if you want to correct for x-y shifts between channels
    #image.set_diameter() # uncomment this line if you want to generate cellpose masks from scratch

    # processing cells + plotting
    cell_type = input("Enter the cell type (e.g., 'Nup98-KDM5A-mEGFP', 'Nup98-KDM5Amut-mEGFP', etc.): ").strip()
    
    # for mEGFP
    cell = Cell(image.h3k4me3, image.puncta, image.masks, key, cell_type) #uncomment this line if cells are Nup98-KDM5A-mEGFP
    
    # for mCherry
    #cell = Cell(image.h3k4me3 / 0.23, image.puncta / 0.23, image.masks, key, cell_type) #uncomment this line if cells are Nup98-KDM5A-mCherry
    
    # cell.separate_cells() # uncomment this line if you want to generate cellpose masks from scratch
    cell.use_existing_masks(masks_loaded) # uncomment this line if you want to use existing cellpose masks stored in masks_loaded
    #cell.make_linear_unscaled_plots() # uncomment this line if you want to generate linear plots with automatic scaling on axes
    #cell.make_log_plots() # uncomment this line if you want to generate log-scaled plots
    cell.make_linear_plots() # uncomment this line if you want to generate linear plots with fixed scaling on axes
    
    print(cell.pearson_rs)
    print(cell.expressions)

    r_value = cell.pearson_rs
    #pearson_dict[key] = r_value # uncomment this line if you want to iteratively add Pearson R values to pearson_dict

    exp_value = cell.expressions
    #expression_dict[key] = exp_value # uncomment this line if you want to iteratively add expression level values to exp_dict